In [ ]:
# RUN THIS FIRST
!pip install -q PyMuPDF==1.23.26 sentence-transformers pinecone-client psutil nltk rouge-score bert-score tiktoken transformers torch scikit-learn
!pip install -q -U google-generativeai
# ===========================================================
# ⚖️ RAG PIPELINE (DistilBERT)
# ===========================================================

import os, re, psutil, time, warnings, csv
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import logging as hf_logging

warnings.filterwarnings("ignore", message="Some weights of the model")
hf_logging.set_verbosity_error()

# ===========================================================
# 1. IMPORTS & SETUP
# ===========================================================
try:
    import fitz  # PyMuPDF
    PDF_LIBRARY = "pymupdf"
    print("✅ Using PyMuPDF for PDF processing")
except ImportError:
    from pypdf import PdfReader
    PDF_LIBRARY = "pypdf"
    print("⚠ Using pypdf fallback")

import nltk
for r in ['wordnet', 'omw-1.4', 'punkt']:
    try: nltk.data.find(f'corpora/{r}')
    except LookupError: nltk.download(r, quiet=True)

from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score
from sklearn.metrics.pairwise import cosine_similarity

# Pinecone setup
try:
    from pinecone import Pinecone, ServerlessSpec
    PINECONE_VERSION = "v3"
    print("✅ Using Pinecone v3+")
except ImportError:
    import pinecone
    PINECONE_VERSION = "v2"
    print("✅ Using Pinecone v2 (legacy)")

print("✅ All imports successful!")

# ===========================================================
# 2. API KEYS
# ===========================================================
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "YOUR_PINECONE_API_KEY_HERE")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "YOUR_GEMINI_API_KEY_HERE")

import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)

if PINECONE_VERSION == "v3":
    pc = Pinecone(api_key=PINECONE_API_KEY)
else:
    pinecone.init(api_key=PINECONE_API_KEY, environment="us-east-1")

print("✅ API clients configured")

# ===========================================================
# 3. PINECONE INDEX (DistilBERT = 768 dims)
# ===========================================================
INDEX_NAME = "distilbert-index-colab"

if PINECONE_VERSION == "v3":
    existing_indexes = [i.name for i in pc.list_indexes()]
    if INDEX_NAME not in existing_indexes:
        pc.create_index(
            name=INDEX_NAME,
            dimension=768,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        print(f"✅ Created new index '{INDEX_NAME}' (768-dim).")
    else:
        print(f"✅ Using existing index '{INDEX_NAME}'.")
    index = pc.Index(INDEX_NAME)
else:
    if INDEX_NAME not in pinecone.list_indexes():
        pinecone.create_index(name=INDEX_NAME, dimension=768, metric="cosine")
    index = pinecone.Index(INDEX_NAME)

print(f"✅ Connected to Pinecone index: {INDEX_NAME}")

# ===========================================================
# 4. GOOGLE DRIVE + FOLDER SELECTION
# ===========================================================
from google.colab import drive
from pathlib import Path

mount_choice = input("Mount Google Drive? (y/n): ").strip().lower()
if mount_choice == "y":
    drive.mount("/content/drive")
    print("✅ Drive mounted.")

folder_type = input("Folder type (Civil/Criminal): ").strip().lower()
if folder_type not in ["civil", "criminal"]:
    print("⚠ Invalid. Defaulting to Civil.")
    folder_type = "civil"

pdf_base_dir = f"/content/drive/MyDrive/RAG&LLM/PDF/{folder_type.upper()}"

process_all = input(f"Process all {folder_type.title()} PDFs? (y/n): ").strip().lower()
if process_all == "y":
    pdf_files = list(Path(pdf_base_dir).rglob("*.pdf"))
else:
    selections = input("Enter folder numbers (e.g., 2001,2005-2008): ").strip()
    pdf_years = []
    for part in selections.split(","):
        if "-" in part:
            start, end = map(int, part.split("-"))
            pdf_years.extend(range(start, end + 1))
        else:
            pdf_years.append(int(part))
    pdf_files = []
    for year in pdf_years:
        folder_path = Path(pdf_base_dir) / str(year)
        if folder_path.exists() and folder_path.is_dir():
            pdf_files.extend(folder_path.rglob("*.pdf"))
        else:
            print(f"⚠ Folder not found: {folder_path}")

if not pdf_files:
    print("⚠ No PDFs found in selected folders. Exiting.")
    exit()
else:
    print(f"✅ Found {len(pdf_files)} PDFs to process.")

# ===========================================================
# 5. PDF EXTRACTION
# ===========================================================
def extract_chunks(pdf_path, chunk_size=200):
    chunks = []
    citation_pattern = re.compile(r"(Section\s\d+[A-Za-z]|Sec\.\s\d+[A-Za-z]|\d+\s?Cr\.?\s?\d+)", re.IGNORECASE)
    try:
        if PDF_LIBRARY == "pymupdf":
            doc = fitz.open(pdf_path)
            text = "\n".join([page.get_text("text") for page in doc])
            doc.close()
        else:
            reader = PdfReader(pdf_path)
            text = "\n".join([page.extract_text() or "" for page in reader.pages])
        text = re.sub(r"[*_]", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        sentences = re.split(r'(?<=[.?!]) +', text)
        buf = ""
        for sent in sentences:
            parts = citation_pattern.split(sent)
            for part in parts:
                seg = part.strip()
                if not seg: continue
                if len(buf.split()) + len(seg.split()) < chunk_size:
                    buf += " " + seg
                else:
                    chunks.append({'text': buf.strip(), 'source': str(pdf_path)})
                    buf = seg
        if buf.strip():
            chunks.append({'text': buf.strip(), 'source': str(pdf_path)})
        return chunks
    except Exception as e:
        print(f"⚠ Error processing {pdf_path}: {e}")
        return []

def process_pdfs(pdf_files):
    all_chunks = []
    for pdf in tqdm(pdf_files, desc="Processing PDFs"):
        all_chunks.extend(extract_chunks(pdf))
    print(f"✅ Total chunks extracted: {len(all_chunks)}")
    return all_chunks

# ===========================================================
# 6. DISTILBERT EMBEDDING
# ===========================================================
def embed_and_store(chunks, model_name="multi-qa-distilbert-cos-v1", batch_size=64):
    if not chunks:
        print("⚠ No chunks found.")
        return None
    print(f"🔹 Using embedding model: {model_name}")
    embedder = SentenceTransformer(model_name)
    texts = [c['text'] for c in chunks]
    vectors = embedder.encode(texts, show_progress_bar=True, batch_size=batch_size, normalize_embeddings=True)
    batch = []
    for i, vec in enumerate(vectors):
        meta = {'text': chunks[i]['text'], 'source': chunks[i]['source']}
        batch.append({'id': str(i), 'values': vec.tolist(), 'metadata': meta})
        if len(batch) >= batch_size:
            index.upsert(vectors=batch)
            batch = []
    if batch:
        index.upsert(vectors=batch)
    print(f"✅ {len(vectors)} chunks embedded & stored in Pinecone.")
    return embedder

# ===========================================================
# 7. RETRIEVAL + GEMINI FLASH GENERATION
# ===========================================================
def retrieve_answer(query, embedder, top_k=7):
    print(f"\n🔹 Query: {query}")
    q_vec = embedder.encode([query])[0]
    res = index.query(vector=q_vec.tolist(), top_k=top_k, include_metadata=True)
    texts = [m['metadata'].get('text', '') for m in res['matches']]
    context = "\n\n".join(texts)

    prompt = f"""You are a legal AI assistant. Using the context below, answer clearly and concisely.

Context:
{context}

Query: {query}

Answer:"""

    try:
        start = time.time()
        model = genai.GenerativeModel("gemini-2.0-flash")
        response = model.generate_content(prompt)
        end = time.time()
        answer = response.text.strip() if response.text else ""
        gen_time = end - start
    except Exception as e:
        print(f"⚠ Gemini Error: {e}")
        answer, gen_time = "", 0
    return answer, texts, gen_time

# ===========================================================
# 8. FULL METRICS + LATENCY
# ===========================================================
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(text):
    return len(enc.encode(text))

def compute_metrics(preds, gts, retrieved_texts_list, q_embeds, embedder,
                    retrieval_times=None, generation_times=None,
                    log_file="metrics_log.csv",
                    embedding_time=0.0, index_size=0):
    df_list = []
    rouge = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    smoothie = SmoothingFunction().method4
    for i, (pred, gt, texts, q_vec) in enumerate(zip(preds, gts, retrieved_texts_list, q_embeds)):
        r = rouge.score(gt, pred)
        b = sentence_bleu([gt.split()], pred.split(), smoothing_function=smoothie)
        m = meteor_score([gt.split()], pred.split())
        P, R, F1 = bert_score([pred], [gt], lang='en', rescale_with_baseline=True)
        retr_vecs = embedder.encode(texts) if embedder and texts else []
        sims = cosine_similarity([q_vec], retr_vecs)[0] if len(retr_vecs) > 0 else []
        if texts:
            P_ctx, R_ctx, F1_ctx = bert_score([pred], [" ".join(texts)], lang="en")
            fcd_value = (1 - float(F1_ctx.mean()))
        else:
            fcd_value = 1.0
        gt_tokens = set(gt.lower().split())
        pred_tokens = set(pred.lower().split())
        common_tokens = gt_tokens & pred_tokens
        faithfulness = len(common_tokens) / len(gt_tokens) if gt_tokens else 0
        terminology_precision = len(common_tokens) / max(1, len(pred_tokens))
        coverage = len(common_tokens) / len(gt_tokens) * 100 if gt_tokens else 0
        bias = abs(len(gt.split()) - len(pred.split())) / max(1, len(gt.split()))
        cosine_sim = float(np.mean(sims)) if len(sims) > 0 else 0
        topk_acc = sum([1 for s in sims if s > 0.8]) / len(sims) if len(sims) > 0 else 0
        retrieval_latency = retrieval_times[i] if retrieval_times else 0
        generation_latency = generation_times[i] if generation_times else 0
        end_latency = retrieval_latency + generation_latency
        throughput = round(1.0 / (end_latency + 1e-6), 2)
        cpu_use = psutil.cpu_percent()
        ram_use = round(psutil.virtual_memory().used / (1024**3), 2)
        context = " ".join(texts)
        context_tokens = count_tokens(context)
        context_len = sum(len(t.split()) for t in texts)
        df_list.append({
            "GT": gt, "Answer": pred,
            "ROUGE-1": r['rouge1'].fmeasure, "ROUGE-2": r['rouge2'].fmeasure, "ROUGE-L": r['rougeL'].fmeasure,
            "BLEU": b, "METEOR": m, "BERT-F1": float(F1[0]),
            "FCD": fcd_value, "Faithfulness": faithfulness, "Bias": bias,
            "Terminology_Precision": terminology_precision, "GroundTruth_Coverage(%)": coverage,
            "Cosine_Sim": cosine_sim, "TopK_Accuracy": topk_acc,
            "Context_Tokens": context_tokens, "Context_Length(words)": context_len,
            "CPU_Usage(%)": cpu_use, "RAM_Usage(GB)": ram_use,
            "Retrieval_Latency(sec)": retrieval_latency, "Generation_Latency(sec)": generation_latency,
            "EndToEnd_Latency(sec)": end_latency, "Throughput(q/s)": throughput
        })
    df = pd.DataFrame(df_list)
    print("\n✅ Evaluation complete. Metrics saved to:", log_file)
    df.to_csv(log_file, index=False)
    return df

# ===========================================================
# 9. MAIN PIPELINE
# ===========================================================
def run_pipeline(pdf_files, queries, gts):
    chunks = process_pdfs(pdf_files)
    embedder = embed_and_store(chunks)
    preds, retrieved_texts_list, q_embeds = [], [], []
    retrieval_times, generation_times = [], []
    for q in queries:
        start_r = time.time()
        ans, texts, gen_t = retrieve_answer(q, embedder)
        end_r = time.time()
        retrieval_times.append(end_r - start_r)
        generation_times.append(gen_t)
        preds.append(ans)
        retrieved_texts_list.append(texts)
        q_embeds.append(embedder.encode([q])[0])
        print(f"⏱ Retrieval Time: {end_r - start_r:.2f}s | Generation Time: {gen_t:.2f}s")
    return compute_metrics(preds, gts, retrieved_texts_list, q_embeds, embedder,
                           retrieval_times, generation_times)

# ===========================================================
# 10. EXAMPLE USAGE
# ===========================================================
civil_queries = [
    "Explain the Balbir Kaur vs Steel Authority of India Ltd. case (2000) and what the Supreme Court decided about compassionate appointments.",
    "Summarize the Datar Switchgears Ltd. vs Tata Finance Ltd. judgment (2000) and its interpretation of the Arbitration and Conciliation Act, 1996.",
    "What was the issue in the All India SC/ST Employees Association vs A. Arthur Jeen case (2001) and how did the Court interpret Article 16(4A)?"
]

civil_gts = [
    "The Balbir Kaur case (2000) held that the Family Benefit Scheme introduced by the Steel Authority of India could not replace the right to compassionate appointment. The Supreme Court ruled that forcing dependents to deposit gratuity and provident fund for monthly benefits violated statutory rights, and compassionate appointment should be available to provide immediate relief to the family of a deceased employee.",
    "In Datar Switchgears Ltd. vs Tata Finance Ltd. (2000), the Supreme Court held that under Section 11(6) of the Arbitration and Conciliation Act, 1996, a party’s right to appoint an arbitrator does not automatically expire after 30 days of demand, as long as the appointment is made before the other party approaches the court. The judgment clarified that such an appointment made before the Section 11 application remains valid.",
    "In All India SC/ST Employees Association vs A. Arthur Jeen (2001), the Supreme Court examined the constitutional validity of Article 16(4A) concerning reservation in promotions for SC/ST employees. The Court held that Article 16(4A) was an enabling provision that allowed, but did not compel, the government to provide reservation in promotion. It reaffirmed that such reservations must be justified through adequate representation and do not automatically apply."
]

df_results = run_pipeline(pdf_files, civil_queries, civil_gts)

print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
for i, row in df_results.iterrows():
    print(f"\n🧾 Query {i+1}:")
    print(f"➡️ {civil_queries[i]}")
    print(f"✅ ROUGE-1: {row['ROUGE-1']:.3f} | ROUGE-2: {row['ROUGE-2']:.3f} | ROUGE-L: {row['ROUGE-L']:.3f}")
    print(f"✅ BLEU: {row['BLEU']:.3f} | METEOR: {row['METEOR']:.3f} | BERT-F1: {row['BERT-F1']:.3f}")
    print(f"✅ FCD: {row['FCD']:.3f} | Faithfulness: {row['Faithfulness']:.3f} | Bias: {row['Bias']:.3f}")
    print(f"✅ Terminology Precision: {row['Terminology_Precision']:.3f} | GT Coverage: {row['GroundTruth_Coverage(%)']:.2f}%")
    print(f"✅ Cosine Sim: {row['Cosine_Sim']:.3f} | TopK Accuracy: {row['TopK_Accuracy']:.2f}")
    print(f"🧩 Context Tokens: {row['Context_Tokens']} | Context Length: {row['Context_Length(words)']}")
    print(f"🖥️ CPU: {row['CPU_Usage(%)']}% | RAM: {row['RAM_Usage(GB)']} GB")
    print(f"⏱️ End-to-End Latency: {row['EndToEnd_Latency(Sec)']:.2f}s | Throughput: {row['Throughput(q/s)']:.2f} q/s")
#final output